# 유저 1 소비 데이터 분석 (2024.01 - 2024.03)

이 노트북은 `data_pre.csv` 데이터를 바탕으로 소비 지표를 집계하고 분석합니다.

In [17]:
import pandas as pd
import numpy as np
import os

# 데이터 로드
file_path = './data_pre.csv'
df = pd.read_csv(file_path)

# 사용 시간 컬럼을 datetime으로 변환
df['사용 시간'] = pd.to_datetime(df['사용 시간'])
df['month'] = df['사용 시간'].dt.month

df.head()

,멤버 id,id,사용 금액,사용 시간,결제 내역,결제 장소 (가맹점 여부),할부 여부,할부 개월,할부 무/유이자 여부,거래 상태 (승인 / 취소),해외 결제,업종 카테고리,결제 방식 (온/오프라인),month
0,1,30001,65000,2024-01-01 10:00:00,SKT통신비,Y,N,0,-,승인,N,생활,자동이체,1
1,1,30002,3714,2024-01-01 09:01:00,이디야,Y,N,0,-,승인,N,식비,온라인 - 카카오페이,1
2,1,30003,8283,2024-01-01 11:12:00,서브웨이,Y,N,0,-,승인,N,식비,오프라인 - 삼성페이,1
3,1,30004,5336,2024-01-01 13:23:00,스타벅스,Y,N,0,-,승인,N,식비,온라인 - 카카오페이,1
4,1,30005,11517,2024-01-01 13:39:00,서브웨이,Y,N,0,-,승인,N,식비,온라인 - 카카오페이,1


## 1. 검증 (Validation)
데이터가 정상적으로 로드되었는지 확인합니다.

In [18]:
# 검증 셀: 데이터가 비어있지 않고, 총 소비 금액이 양수여야 함
assert len(df) > 0, "데이터가 존재하지 않습니다."
assert df['사용 금액'].sum() > 0, "총 소비 금액이 0원 이하입니다."
print("✅ 데이터 로드 및 기본 검증 통과")

✅ 데이터 로드 및 기본 검증 통과


## 2. 전체 소비 요약

In [19]:
summary = {
    "총 소비 금액": df['사용 금액'].sum(),
    "총 결제 건수": len(df),
    "평균 결제 금액": round(df['사용 금액'].mean(), 0)
}

summary_df = pd.DataFrame([summary])
summary_df

,총 소비 금액,총 결제 건수,평균 결제 금액
0,5152575,463,11129.0


## 3. 카테고리별 지출 지표

In [20]:
category_stats = df.groupby('업종 카테고리')['사용 금액'].agg(['sum', 'mean', 'count']).sort_values(by='sum', ascending=False)
category_stats.columns = ['총 지출액', '평균 지출액', '결제 건수']

# 비중 계산
total_sum = category_stats['총 지출액'].sum()
category_stats['지출 비중(%)'] = round((category_stats['총 지출액'] / total_sum) * 100, 2)

category_stats

,총 지출액,평균 지출액,결제 건수,지출 비중(%)
업종 카테고리,,,,
식비,3368156,10830.083601,311,65.37
쇼핑,880961,11746.146667,75,17.10
의료,488332,20347.166667,24,9.48
교통,220126,4402.520000,50,4.27
생활,195000,65000.000000,3,3.78


## 4. 월별 지출 추이

In [21]:
month_stats = df.groupby('month')['사용 금액'].sum().reset_index()
month_stats.columns = ['월', '월별 총 지출액']
month_stats

,월,월별 총 지출액
0,1,1894343
1,2,1596338
2,3,1661894


## 5. 지출액 상위 5개 가맹점

In [22]:
top_merchants = df.groupby('결제 내역')['사용 금액'].sum().sort_values(ascending=False).head(5)
top_merchants_df = top_merchants.reset_index()
top_merchants_df.columns = ['가맹점', '총 지출액']
top_merchants_df

,가맹점,총 지출액
0,쿠팡이츠,927848
1,배달의민족,747609
2,쿠팡,535707
3,맥도날드,458662
4,치과,423106
